In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path
import numpy as np
import os
import shutil
import json
import zipfile

sys.path.append(str(Path().resolve().parents[0]))

# Local src

from src.df_overview import df_overview
from src.generate_metadata import generate_metadata

print('Done')

Done


In [7]:
path_rfm_csv = '../data/02_processed/rfm_retail.csv'
path_base_csv = '../data/02_processed/base_retail.csv'

df_rfm = pd.read_csv(path_rfm_csv)
df_base = pd.read_csv(path_base_csv)

In [12]:
df_base

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,revenue,sales_only,returns_only
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.40,83.40,0.0
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,81.00,0.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,81.00,0.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.80,100.80,0.0
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.00,30.00,0.0
...,...,...,...,...,...,...,...,...,...,...,...
793750,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680,France,10.20,10.20,0.0
793751,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680,France,12.60,12.60,0.0
793752,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680,France,16.60,16.60,0.0
793753,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680,France,16.60,16.60,0.0


In [9]:
df_base['sales_only'] = df_base['revenue'].clip(lower=0)
df_base['returns_only'] = df_base['revenue'].clip(upper=0).abs()

ratios = df_base.groupby('customer_id').agg(
    total_sales=('sales_only', 'sum'),
    total_returns=('returns_only', 'sum')
)

ratios['return_ratio'] = ratios['total_returns'] / ratios['total_sales']
ratios['return_ratio'] = ratios['return_ratio'].fillna(0)
ratios.reset_index()
ratios
df_rfm = df_rfm.set_index('customer_id')
df_rfm = df_rfm.merge(ratios[['return_ratio']], left_index=True, right_index=True, how='left')

In [13]:
df_rfm.describe()

,frequency,monetary,recency,r_score,f_score,m_score,return_ratio
count,5847.000000,5847.000000,5847.000000,5847.000000,5847.000000,5847.000000,5847.000000
mean,6.259107,2889.555760,199.161621,2.999487,2.999487,2.999487,0.022750
std,12.754038,14147.743908,208.504831,1.414395,1.414395,1.414395,0.081171
min,1.000000,2.950000,0.000000,1.000000,1.000000,1.000000,0.000000
25%,1.000000,339.550000,24.000000,2.000000,2.000000,2.000000,0.000000
50%,3.000000,856.010000,94.000000,3.000000,3.000000,3.000000,0.000000
75%,7.000000,2239.355000,378.000000,4.000000,4.000000,4.000000,0.014676
max,373.000000,580987.040000,738.000000,5.000000,5.000000,5.000000,1.000000


In [ ]:
pd_to_csv_path = os.path.join('..', 'data', '03_featured', 'featured_rfm_retail.csv')

os.makedirs(os.path.dirname(pd_to_csv_path), exist_ok=True)
df_rfm.to_csv(pd_to_csv_path, index=False)

generate_metadata(
    df=df_rfm_sql,
    csv_path=pd_to_csv_path
)

print(f"Saved to: {pd_to_csv_path}")

Saved to: ../data/03_featured/featured_rfm_retail.csv


customer_age_days = snapshot_date - first_purchase_date
avg_interpurchase_time = customer_age_days / (frequency - 1)
avg_order_value